### Kaggle Authentication
Join the competition, generate an api token from settings page and replace this with your own token.

In [1]:
!export KAGGLE_API_TOKEN=KGAT_62cc08268d9cba3a4c8a7882908eed28

### Setup Kaggle Access Token
Setup Kaggle Credentials Write access token to disk for kagglehub authentication.

In [2]:
!mkdir -p ~/.kaggle && echo KGAT_62cc08268d9cba3a4c8a7882908eed28 > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

### Download Competition Data
Download Competition Data Pull the competition dataset via kagglehub.

In [3]:
import kagglehub

path = kagglehub.competition_download("multi-view-pig-posture-recognition")

100%|██████████| 1.82G/1.82G [01:47<00:00, 18.2MB/s]

Extracting files...


### Download Pretrained Fold Weights for inference
Download the 3-fold DINOv2-G checkpoints from Kaggle.

**Don't run this cell if you want to train from zero. this checkpoint scored 0.886 in private leaderboard. if you want to reproduce the score, run this cell and set inference_only parameter to True in Args.**

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ramazanturann/pig-posture-prediction-dinov2g-3fold")

print("Path to dataset files:", path)

100%|██████████| 11.8G/11.8G [11:59<00:00, 17.6MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ramazanturann/pig-posture-prediction-dinov2g-3fold/versions/10


In [5]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.7 MB/s eta 0:00:00


### Imports, Paths & Class Definitions
All imports, dataset paths, class names, model registry, and flip mapping.

In [15]:
#!/usr/bin/env python3

import os
import gc
import ast
import time
import math
import random
import warnings
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import albumentations as A
from albumentations.pytorch import ToTensorV2

import timm
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

DATA_ROOT  = Path("/root/.cache/kagglehub/competitions/multi-view-pig-posture-recognition/multiview_pig_posture_recognition")

TRAIN1_DIR = DATA_ROOT / "train1_images"
TRAIN2_DIR = DATA_ROOT / "train2_images"
TEST_DIR   = DATA_ROOT / "test_images"
TRAIN1_CSV = DATA_ROOT / "train1.csv"
TRAIN2_CSV = DATA_ROOT / "train2.csv"
TEST_CSV   = DATA_ROOT / "test.csv"
OUT_DIR    = Path("/root/.cache/kagglehub/datasets/ramazanturann/pig-posture-prediction-dinov2g-3fold/versions/11")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 5
CLASS_NAMES = [
    "Lateral_lying_left",
    "Lateral_lying_right",
    "Sitting",
    "Standing",
    "Sternal_lying",
]
HFLIP_MAP = [1, 0, 2, 3, 4]

MODEL_CONFIGS = [
    ("dinov2_g", "dinov2_g", 224, 8, 21),
]

DINOV2_TIMM_MAP = {
    "dinov2_s": "vit_small_patch14_reg4_dinov2.lvd142m",
    "dinov2_b": "vit_base_patch14_reg4_dinov2.lvd142m",
    "dinov2_l": "vit_large_patch14_reg4_dinov2.lvd142m",
    "dinov2_g": "vit_giant_patch14_reg4_dinov2.lvd142m",
    "eva_g":    "eva_giant_patch14_336.m30m_ft_in22k_in1k",
    "pe_gigantic": "vit_pe_core_gigantic_patch14_448.fb",
}

### Hyperparameters & Device Setup
Training config, learning rate schedule params, and GPU detection.

In [16]:
SEED              = 42
DINOV2_LR         = 5e-5
BASE_LR           = 2e-4
LLRD_DECAY        = 0.85
MIN_LR            = 2e-6
WEIGHT_DECAY      = 1e-2
WARMUP_EPOCHS     = 3
FOCAL_GAMMA       = 2.5
PAD_FRACTION      = 0.15
USE_AMP           = True
NUM_WORKERS       = 2
TTA_ENABLED       = True
MIXUP_ALPHA       = 0.2
EARLY_STOP_PAT    = 8

N_FOLDS = 3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

Device: cuda
GPU   : NVIDIA A100-SXM4-40GB
VRAM  : 42.4 GB


## Utility Functions
Seed, bbox crop, camera type parsing, class/sample weights, and checkpoint I/O.

In [17]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def parse_bbox(s):
    return list(map(float, ast.literal_eval(s)))


def crop_and_resize(img_path, x, y, w, h, img_size, pad=PAD_FRACTION):
    img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    if img is None:
        return np.zeros((img_size, img_size, 3), dtype=np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    px, py = int(w * pad), int(h * pad)
    x1 = max(0, int(x) - px)
    y1 = max(0, int(y) - py)
    x2 = min(W, int(x + w) + px)
    y2 = min(H, int(y + h) + py)
    crop = img[y1:y2, x1:x2].copy()
    del img
    if crop.size == 0:
        crop = np.zeros((img_size, img_size, 3), dtype=np.uint8)
    return cv2.resize(crop, (img_size, img_size), interpolation=cv2.INTER_LINEAR)


def extract_camera_id(image_id):
    parts = image_id.split("_")
    return "_".join(parts[:3]) if len(parts) >= 3 else image_id


def extract_cam_type(image_id):
    parts = image_id.split("_")
    return parts[1] if len(parts) >= 2 else "unknown"


def compute_class_weights(labels):
    counts = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
    weights = 1.0 / (counts + 1e-6)
    return torch.tensor(weights / weights.sum() * NUM_CLASSES, dtype=torch.float32)


def compute_sample_weights(labels, sternal_boost=3.0, lateral_boost=1.5):
    counts = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
    weights = 1.0 / (counts + 1e-6)
    weights[4] *= sternal_boost
    weights[0] *= lateral_boost
    weights[1] *= lateral_boost
    sample_weights = weights[labels]
    return sample_weights


def is_dinov2(model_key):
    return model_key in DINOV2_TIMM_MAP


def build_img_info(df):
    img_info = (
        df.groupby("image_id")
        .agg(
            dominant_class=("class_id", lambda s: s.mode()[0]),
        )
        .reset_index()
    )
    img_info["cam_type"] = img_info["image_id"].apply(extract_cam_type)
    img_info["stratum"]  = (
        img_info["cam_type"] + "_" + img_info["dominant_class"].astype(str)
    )
    rare = img_info["stratum"].value_counts()
    rare = rare[rare < N_FOLDS].index
    img_info.loc[img_info["stratum"].isin(rare), "stratum"] = (
        img_info.loc[img_info["stratum"].isin(rare), "dominant_class"].astype(str)
    )
    return img_info


def get_full_ckpt_path(set_label, tag):
    return OUT_DIR / f"{set_label}_{tag}_full_ckpt.pt"


def get_best_ema_path(set_label, tag):
    return OUT_DIR / f"{set_label}_{tag}_best_ema.pt"


def save_full_checkpoint(path, model, ema_model, optimizer, scheduler, scaler,
                         epoch, best_f1, stale):
    torch.save({
        "epoch":           epoch,
        "best_f1":         best_f1,
        "stale":           stale,
        "model_state":     model.state_dict(),
        "ema_state":       ema_model.state_dict(),
        "scheduler_epoch": scheduler.current_epoch,
        "scaler_state":    scaler.state_dict(),
    }, path)
    print(f"    ✓ Checkpoint saved: {path.name}")


def load_full_checkpoint(path, model, ema_model, scheduler, scaler):
    print(f"\n  ↩ Loading checkpoint: {path.name}")
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    ema_model.load_state_dict(ckpt["ema_state"])
    scaler.load_state_dict(ckpt["scaler_state"])
    scheduler.current_epoch = ckpt["scheduler_epoch"]
    start_epoch = ckpt["epoch"] + 1
    best_f1     = ckpt["best_f1"]
    stale       = ckpt["stale"]
    print(f"  ✓ Resuming from epoch {ckpt['epoch']} | Best F1: {best_f1:.4f} | Stale: {stale}")
    return start_epoch, best_f1, stale

### Dataset & Augmentation
Class-specific augmentation policies, PigPostureDataset, and MixUp.

In [18]:
def _color_aug():
    return A.OneOf([
        A.ColorJitter(brightness=0.40, contrast=0.40, saturation=0.30, hue=0.12, p=1.0),
        A.HueSaturationValue(hue_shift_limit=25, sat_shift_limit=40, val_shift_limit=30, p=1.0),
        A.CLAHE(clip_limit=5.0, tile_grid_size=(8, 8), p=1.0),
        A.RandomGamma(gamma_limit=(70, 140), p=1.0),
    ], p=0.80)


def _noise_aug():
    return A.OneOf([
        A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
        A.ISONoise(color_shift=(0.01, 0.06), intensity=(0.1, 0.5), p=1.0),
        A.MotionBlur(blur_limit=7, p=1.0),
        A.GaussianBlur(blur_limit=(3, 9), p=1.0),
    ], p=0.35)


def _dropout_aug(img_size):
    return A.CoarseDropout(
        max_holes=6, max_height=img_size // 5, max_width=img_size // 5,
        min_holes=1, fill_value=0, p=0.40,
    )


def _normalize():
    return A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)


def get_train_transform_for_class(img_size, label):
    base_geometric = [
        A.ShiftScaleRotate(
            shift_limit=0.07, scale_limit=0.15, rotate_limit=20,
            border_mode=cv2.BORDER_REFLECT_101, p=0.70,
        ),
        A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.20),
        A.ElasticTransform(alpha=60, sigma=6, p=0.15),
    ]

    if label == 4:
        class_aug = [
            A.RandomRotate90(p=0.90),
            A.Rotate(limit=180, border_mode=cv2.BORDER_REFLECT_101, p=0.70),
            A.Perspective(scale=(0.08, 0.25), p=0.65),
        ]
    elif label in (0, 1):
        class_aug = [
            A.RandomRotate90(p=0.10),
            A.Perspective(scale=(0.05, 0.18), p=0.60),
            A.ElasticTransform(alpha=40, sigma=5, p=0.25),
        ]
    elif label == 2:
        class_aug = [
            A.RandomRotate90(p=0.50),
            A.Perspective(scale=(0.12, 0.32), p=0.80),
            A.ShiftScaleRotate(
                shift_limit=0.05, scale_limit=0.25, rotate_limit=45,
                border_mode=cv2.BORDER_REFLECT_101, p=0.70,
            ),
        ]
    else:
        class_aug = [
            A.RandomRotate90(p=0.25),
            A.Perspective(scale=(0.08, 0.20), p=0.60),
        ]

    return A.Compose(
        class_aug + base_geometric + [
            _color_aug(),
            _noise_aug(),
            _dropout_aug(img_size),
            _normalize(),
            ToTensorV2(),
        ]
    )


def get_val_transform(img_size):
    return A.Compose([
        _normalize(),
        ToTensorV2(),
    ])


class PigPostureDataset(Dataset):
    def __init__(self, df, img_dir, transform, img_size, mode="train",
                 hflip_prob=0.5, use_class_aug=False):
        self.df            = df.reset_index(drop=True)
        self.img_dir       = img_dir
        self.transform     = transform
        self.img_size      = img_size
        self.mode          = mode
        self.hflip_prob    = hflip_prob if mode == "train" else 0.0
        self.use_class_aug = use_class_aug

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        x, y, w, h = parse_bbox(row["bbox"])
        label  = int(row["class_id"]) if self.mode != "test" else -1
        do_flip = self.hflip_prob > 0 and random.random() < self.hflip_prob

        if self.mode == "train" and label != -1:
            if label == 2:
                pad = 0.22
            elif label in (0, 1, 4):
                pad = 0.08
            else:
                pad = 0.18
        else:
            pad = PAD_FRACTION

        crop = crop_and_resize(
            self.img_dir / row["image_id"], x, y, w, h, self.img_size, pad=pad
        )

        if do_flip:
            crop  = cv2.flip(crop, 1)
            label = HFLIP_MAP[label] if self.mode != "test" else label

        if self.mode == "train" and self.use_class_aug and label != -1:
            transform = get_train_transform_for_class(self.img_size, label)
        else:
            transform = self.transform

        tensor = transform(image=crop)["image"]
        del crop

        if self.mode == "test":
            return tensor, row["row_id"]
        return tensor, label


def mixup_batch(imgs, labels, alpha=MIXUP_ALPHA, num_classes=NUM_CLASSES):
    if alpha <= 0 or random.random() > 0.3:
        return imgs, F.one_hot(labels, num_classes).float()
    lam = max(np.random.beta(alpha, alpha), 1 - np.random.beta(alpha, alpha))
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    mixed = lam * imgs + (1 - lam) * imgs[idx]
    la = F.one_hot(labels, num_classes).float()
    lb = F.one_hot(labels[idx], num_classes).float()
    return mixed, lam * la + (1 - lam) * lb

 ### Loss Functions
 Focal Loss and PairwisePenaltyLoss targeting hard confusion pairs.

In [19]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha     = alpha
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        log_prob = F.log_softmax(logits, dim=-1)
        prob     = torch.exp(log_prob)
        if targets.dim() == 1:
            p_t = prob.gather(1, targets.unsqueeze(1)).squeeze(1)
            fw  = (1 - p_t) ** self.gamma
            ce  = F.nll_loss(log_prob, targets, reduction="none")
            if self.alpha is not None:
                ce = self.alpha.to(logits.device)[targets] * ce
        else:
            p_t = (prob * targets).sum(dim=-1)
            fw  = (1 - p_t) ** self.gamma
            ce  = -(targets * log_prob).sum(dim=-1)
            if self.alpha is not None:
                ce = self.alpha.to(logits.device)[targets.argmax(1)] * ce
        loss = fw * ce
        return loss.mean() if self.reduction == "mean" else loss.sum()


class PairwisePenaltyLoss(nn.Module):
    PENALTY_PAIRS = {
        (4, 0): 3.5,
        (4, 1): 3.5,
        (0, 4): 2.5,
        (1, 4): 2.5,
        (4, 2): 3.0,
        (2, 4): 3.0,
        (0, 1): 1.5,
        (1, 0): 1.5,
    }

    def __init__(self, focal_alpha=None, focal_gamma=2.5):
        super().__init__()
        self.focal = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)
        penalty = torch.ones(NUM_CLASSES, NUM_CLASSES)
        for (pred_c, true_c), w in self.PENALTY_PAIRS.items():
            penalty[pred_c, true_c] = w
        self.register_buffer("penalty_matrix", penalty)

    def forward(self, logits, targets):
        focal_loss = self.focal(logits, targets)
        if targets.dim() == 1:
            preds        = logits.argmax(dim=1)
            weights      = self.penalty_matrix[preds, targets]
            penalty_loss = focal_loss * weights.mean()
            return 0.65 * focal_loss + 0.35 * penalty_loss
        return focal_loss

 ### Model, Optimizer & Scheduler
 DINOv2 model builder, layer-wise LR decay optimizer, cosine warmup scheduler, and EMA.

In [20]:
def build_model(model_key, num_classes=NUM_CLASSES):
    if is_dinov2(model_key):
        timm_name = DINOV2_TIMM_MAP[model_key]
        model = timm.create_model(
            timm_name,
            pretrained=True,
            num_classes=num_classes,
            img_size=224,
        )
        n = sum(p.numel() for p in model.parameters()) / 1e6
        print(f"  [DINOv2] {timm_name} → {n:.1f}M param")
        return model.to(DEVICE)
    elif "maxvit" in model_key:
        model = timm.create_model(model_key, pretrained=True,
                                  num_classes=num_classes, drop_path_rate=0.2)
    elif "swinv2" in model_key:
        model = timm.create_model(model_key, pretrained=True,
                                  num_classes=num_classes, drop_path_rate=0.2)
    elif "eva02" in model_key:
        model = timm.create_model(model_key, pretrained=True,
                                  num_classes=num_classes, drop_path_rate=0.1)
    elif "vit_pe_core" in model_key:
        model = timm.create_model(
            model_key, pretrained=True, num_classes=num_classes,
            img_size=448, drop_path_rate=0.15,
        )
    else:
        model = timm.create_model(model_key, pretrained=True,
                                  num_classes=num_classes,
                                  drop_rate=0.3, drop_path_rate=0.2)

    n = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  [{model_key}] {n:.1f}M param")
    return model.to(DEVICE)


def get_llrd_optimizer(model, model_key, base_lr, weight_decay, decay=LLRD_DECAY):
    if not is_dinov2(model_key):
        return optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=base_lr, weight_decay=weight_decay,
        )

    param_groups = []

    head_params = [p for n, p in model.named_parameters()
                   if "head" in n and p.requires_grad]
    if head_params:
        param_groups.append({
            "params":     head_params,
            "lr":         base_lr,
            "base_lr":    base_lr,
            "group_name": "head",
        })
        print(f"  LLRD head: lr={base_lr:.1e}")

    if hasattr(model, "blocks"):
        blocks     = list(model.blocks)
        num_blocks = len(blocks)
        for i, block in enumerate(reversed(blocks)):
            lr_i = base_lr * (decay ** (i + 1))
            block_params = [p for p in block.parameters() if p.requires_grad]
            if block_params:
                param_groups.append({
                    "params":     block_params,
                    "lr":         lr_i,
                    "base_lr":    lr_i,
                    "group_name": f"block_{num_blocks - 1 - i}",
                })
        if num_blocks > 0:
            print(f"  LLRD blocks: {num_blocks} blocks | "
                  f"last={base_lr * decay:.2e} → first={base_lr * decay**num_blocks:.2e}")
    else:
        num_blocks = 0

    known_param_ids = {id(p) for g in param_groups for p in g["params"]}
    other_params = [p for p in model.parameters()
                    if p.requires_grad and id(p) not in known_param_ids]
    if other_params:
        lr_other = base_lr * (decay ** (num_blocks + 2))
        param_groups.append({
            "params":     other_params,
            "lr":         lr_other,
            "base_lr":    lr_other,
            "group_name": "patch_embed_norm",
        })
        print(f"  LLRD patch/norm: lr={lr_other:.2e}")

    total_trainable = sum(p.numel() for g in param_groups for p in g["params"]) / 1e6
    print(f"  LLRD total trainable: {total_trainable:.1f}M params, {len(param_groups)} groups")

    return optim.AdamW(param_groups, weight_decay=weight_decay)


class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_epochs, total_epochs, base_lr, min_lr=1e-6):
        self.optimizer     = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs  = total_epochs
        self.base_lr       = base_lr
        self.min_lr        = min_lr
        self.current_epoch = 0

    def step(self):
        self.current_epoch += 1
        e = self.current_epoch
        if e <= self.warmup_epochs:
            scale = e / max(self.warmup_epochs, 1)
        else:
            p     = (e - self.warmup_epochs) / max(self.total_epochs - self.warmup_epochs, 1)
            scale = self.min_lr / self.base_lr + 0.5 * (1 - self.min_lr / self.base_lr) * (
                1 + math.cos(math.pi * p)
            )

        for pg in self.optimizer.param_groups:
            group_base = pg.get("base_lr", self.base_lr)
            pg["lr"]   = max(group_base * scale, self.min_lr)

        return self.optimizer.param_groups[0]["lr"]


class ModelEMA:
    def __init__(self, model, decay=0.9999):
        self.decay     = decay
        self.ema_model = deepcopy(model).eval()
        for p in self.ema_model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for ep, mp in zip(self.ema_model.parameters(), model.parameters()):
            ep.data.mul_(self.decay).add_(mp.data.to(ep.device), alpha=1 - self.decay)
        for eb, mb in zip(self.ema_model.buffers(), model.buffers()):
            eb.copy_(mb.to(eb.device))

### Training & Evaluation Loop
Single-epoch train step, validation, and full train_model orchestrator with early stopping.

In [21]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, ema=None):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    pbar = tqdm(loader, desc="  train", leave=False, ncols=110)
    for imgs, labels in pbar:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        imgs, soft = mixup_batch(imgs, labels)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP and DEVICE.type == "cuda"):
            logits = model(imgs)
            loss   = criterion(logits, soft)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        if ema is not None:
            ema.update(model)
        bs          = imgs.size(0)
        total_loss += loss.item() * bs
        hard        = soft.argmax(1) if soft.dim() > 1 else labels
        correct    += (logits.argmax(1) == hard).sum().item()
        n          += bs
        pbar.set_postfix({"loss": f"{total_loss/n:.4f}", "acc": f"{correct/n:.3f}"})
    return total_loss / n, correct / n


def swap_lr(p):
    s = p.copy()
    s[:, 0] = p[:, 1]
    s[:, 1] = p[:, 0]
    return s


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in tqdm(loader, desc="  val  ", leave=False, ncols=110):
        imgs = imgs.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP and DEVICE.type == "cuda"):
            logits = model(imgs)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
    macro = f1_score(all_labels, all_preds, average="macro",  zero_division=0)
    per_c = f1_score(all_labels, all_preds, average=None,     zero_division=0)
    return macro, per_c


def train_model(
        model_key, tag, img_size, batch_size, total_epochs,
        train_df, val_df, img_dir, set_label, class_weights,
        resume=False,
        resume_if_exists=False,
        tag_suffix="",
):
    actual_tag = tag + tag_suffix
    print(f"\n{'=' * 70}")
    lr = DINOV2_LR if (is_dinov2(model_key) or "eva02" in model_key) else BASE_LR

    print(f"  {set_label} → {actual_tag}  "
          f"[{img_size}px, bs={batch_size}, ep={total_epochs}, head_lr={lr:.0e}]")
    print(f"{'=' * 70}")

    full_ckpt_path = get_full_ckpt_path(set_label, actual_tag)
    best_ema_path  = get_best_ema_path(set_label, actual_tag)

    train_ds = PigPostureDataset(
        train_df, img_dir,
        transform=get_val_transform(img_size),
        img_size=img_size,
        mode="train",
        use_class_aug=True,
    )
    val_ds = PigPostureDataset(
        val_df, img_dir, get_val_transform(img_size), img_size, mode="val"
    )

    sampler = WeightedRandomSampler(
        compute_sample_weights(train_df["class_id"].values),
        len(train_df), replacement=True,
    )
    train_dl = DataLoader(
        train_ds, batch_size=batch_size, sampler=sampler,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
        persistent_workers=NUM_WORKERS > 0,
    )
    val_dl = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
    )

    model = build_model(model_key)
    ema   = ModelEMA(model, decay=0.9999)

    criterion = PairwisePenaltyLoss(focal_alpha=class_weights, focal_gamma=FOCAL_GAMMA)
    optimizer = get_llrd_optimizer(model, model_key, lr, WEIGHT_DECAY, decay=LLRD_DECAY)
    scheduler = WarmupCosineScheduler(optimizer, WARMUP_EPOCHS, total_epochs, lr, MIN_LR)
    scaler    = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    start_epoch = 1
    best_f1     = 0.0
    stale       = 0

    should_resume = resume or (resume_if_exists and full_ckpt_path.exists())
    if should_resume:
        if full_ckpt_path.exists():
            start_epoch, best_f1, stale = load_full_checkpoint(
                full_ckpt_path, model, ema.ema_model, scheduler, scaler
            )
            stale = 0
            print(f"  ✓ Stale counter reset after resume")
        elif resume:
            raise FileNotFoundError(f"Checkpoint not found: {full_ckpt_path}")
        else:
            print("  ✓ No checkpoint found, starting from scratch.")
    else:
        print(f"  ✓ Starting from scratch (epoch 1/{total_epochs})")

    if start_epoch > total_epochs:
        print(f"  ✓ Training already complete")
        if best_ema_path.exists():
            ema.ema_model.load_state_dict(
                torch.load(best_ema_path, map_location=DEVICE, weights_only=False)
            )
        val_probs = get_tta_probs_val(ema.ema_model, val_df, img_dir, img_size, batch_size)
        return ema.ema_model, val_probs, val_df["class_id"].values, best_f1

    best_ema_state = None
    if should_resume and best_ema_path.exists():
        best_ema_state = torch.load(best_ema_path, map_location=DEVICE, weights_only=False)
        print("  ✓ Previous best EMA loaded")

    for epoch in range(start_epoch, total_epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc      = train_one_epoch(model, train_dl, optimizer, criterion, scaler, ema)
        val_f1, per_class_f1 = evaluate(ema.ema_model, val_dl)
        lr_cur               = scheduler.step()
        elapsed              = time.time() - t0

        gpu_str = ""
        if DEVICE.type == "cuda":
            used      = torch.cuda.memory_allocated() / 1e9
            total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
            gpu_str   = f" | GPU {used:.1f}/{total_mem:.0f}GB"

        print(
            f"  Ep {epoch:02d}/{total_epochs} | lr={lr_cur:.1e} | "
            f"tr_loss={tr_loss:.4f} acc={tr_acc:.3f} | "
            f"val_F1={val_f1:.4f}{gpu_str} | {elapsed:.0f}s"
        )
        print(
            f"    Sit={per_class_f1[2]:.3f} Left={per_class_f1[0]:.3f} "
            f"Right={per_class_f1[1]:.3f} Stand={per_class_f1[3]:.3f} "
            f"Stern={per_class_f1[4]:.3f}"
        )

        if val_f1 > best_f1:
            best_f1        = val_f1
            best_ema_state = deepcopy(ema.ema_model.state_dict())
            torch.save(best_ema_state, best_ema_path)
            print(f"    ★ Best val F1={best_f1:.4f}")
            stale = 0
        else:
            stale += 1
            if stale >= EARLY_STOP_PAT:
                print(f"    ⏹ Early stop (patience={EARLY_STOP_PAT})")
                save_full_checkpoint(
                    full_ckpt_path, model, ema.ema_model, optimizer,
                    scheduler, scaler, epoch, best_f1, stale,
                )
                break

        save_full_checkpoint(
            full_ckpt_path, model, ema.ema_model, optimizer,
            scheduler, scaler, epoch, best_f1, stale,
        )

    if best_ema_state is not None:
        ema.ema_model.load_state_dict(best_ema_state)
    elif best_ema_path.exists():
        ema.ema_model.load_state_dict(
            torch.load(best_ema_path, map_location=DEVICE, weights_only=False)
        )
    ema.ema_model.eval()

    val_probs  = get_tta_probs_val(ema.ema_model, val_df, img_dir, img_size, batch_size)
    val_labels = val_df["class_id"].values
    print(f"  ✓ Best val F1: {best_f1:.4f}")

    del model, optimizer, scaler, train_ds, val_ds, train_dl
    gc.collect()
    torch.cuda.empty_cache()

    return ema.ema_model, val_probs, val_labels, best_f1

### Inference, TTA & Submission
Probability extraction, 8x TTA, fold ensembling, and CSV export.

In [22]:
@torch.no_grad()
def get_probs(model, loader, with_ids=False):
    model.eval()
    all_probs, all_ids = [], []
    for batch in tqdm(loader, desc="  infer", leave=False, ncols=110):
        if with_ids:
            imgs, ids = batch
            all_ids.extend(list(ids))
        else:
            imgs = batch[0]
        imgs = imgs.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP and DEVICE.type == "cuda"):
            logits = model(imgs)
        all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    probs = np.vstack(all_probs)
    return (probs, all_ids) if with_ids else probs


def _make_tta_transforms(img_size):
    N = A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    T = ToTensorV2()
    transforms = []
    for angle in [0, 90, 180, 270]:
        for do_flip in [False, True]:
            aug_list = []
            if angle != 0:
                aug_list.append(A.Rotate(limit=(angle, angle), p=1.0))
            if do_flip:
                aug_list.append(A.HorizontalFlip(p=1.0))
            aug_list += [N, T]
            transforms.append((A.Compose(aug_list), do_flip))
    return transforms


@torch.no_grad()
def get_tta_probs(model, df_test, img_dir, img_size, batch_size):
    model.eval()
    all_tta_transforms = _make_tta_transforms(img_size)
    print(f"  TTA: {len(all_tta_transforms)} augmentations × {len(df_test)} instances")
    all_probs   = []
    row_ids_out = None
    for i, (aug, needs_swap) in enumerate(all_tta_transforms):
        ds = PigPostureDataset(df_test, img_dir, aug, img_size, mode="test")
        dl = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
        if i == 0:
            probs, row_ids_out = get_probs(model, dl, with_ids=True)
        else:
            probs = get_probs(model, dl, with_ids=False)
        if needs_swap:
            probs = swap_lr(probs)
        all_probs.append(probs)
    return np.mean(all_probs, axis=0), row_ids_out


@torch.no_grad()
def get_tta_probs_val(model, val_df, img_dir, img_size, batch_size):
    model.eval()
    all_tta_transforms = _make_tta_transforms(img_size)
    all_probs = []
    for aug, needs_swap in all_tta_transforms:
        ds    = PigPostureDataset(val_df, img_dir, aug, img_size, mode="val")
        dl    = DataLoader(ds, batch_size=batch_size, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
        probs = get_probs(model, dl, with_ids=False)
        if needs_swap:
            probs = swap_lr(probs)
        all_probs.append(probs)
    return np.mean(all_probs, axis=0)


def weighted_ensemble(probs_list):
    total_w = sum(w for _, w in probs_list)
    return sum(p * w for p, w in probs_list) / total_w


def save_submission(probs, row_ids, path, set_label="", approach=""):
    preds  = probs.argmax(axis=1)
    df_sub = pd.DataFrame({"row_id": row_ids, "class_id": preds.astype(int)})
    df_sub.to_csv(path, index=False)
    dist = {CLASS_NAMES[int(k)]: int(v)
            for k, v in dict(pd.Series(preds).value_counts().sort_index()).items()}
    print(f"\n  [{set_label}] {approach} → {Path(path).name}")
    print(f"  Distribution: {dist}")
    for cls_id, thr in [(0, 50), (1, 50), (2, 100)]:
        if int((preds == cls_id).sum()) < thr:
            print(f"  ⚠ {CLASS_NAMES[cls_id]} very low count: {int((preds == cls_id).sum())}")

###  K-Fold Pipeline, Inference-Only & Entry Point
Full K-Fold training pipeline, inference-only mode, Args config class, and main().

Peak of the model is around 27-28 epoch. i set n_folds parameter to 3 and epoch to 22 because i there was 14~ hours remaining until competitions ends. this model scored 0.871 in private leaderboard with 27 epochs without k-fold. 3 folds and 22 epochs boosted score to 0.886 which is a really good improvement for this competition. 3 folds and 22 epochs full pipeline including inference takes about 12-14 hours with A100 40GB. i believe 4 or 5 folds with 27 epochs would get 1st place in the private leaderboard.

In [ ]:
def run_kfold_pipeline(
        train_csv, img_dir, set_label,
        total_epochs_override=None,
        resume=False,
        resume_if_exists=False,
        specific_tags=None,
        n_folds=N_FOLDS,
):
    print(f"\n{'#' * 70}")
    print(f"#  {set_label} K-FOLD PIPELINE  (v5, {n_folds}-fold)")
    print(f"{'#' * 70}")

    set_seed(SEED)
    df_train = pd.read_csv(train_csv)
    df_test  = pd.read_csv(TEST_CSV)

    print(f"\nTraining instances: {len(df_train):,}")
    for cls_id, name in enumerate(CLASS_NAMES):
        n = (df_train["class_id"] == cls_id).sum()
        print(f"  {cls_id}: {name:<25} {n:5d} ({n/len(df_train)*100:.1f}%)")

    img_info = build_img_info(df_train)
    print(f"\nTotal images: {len(img_info):,} | Strata count: {img_info['stratum'].nunique()}")

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

    configs_to_run = [
        c for c in MODEL_CONFIGS
        if specific_tags is None or c[1] in specific_tags
    ]

    for model_key, tag, img_size, batch_size, default_epochs in configs_to_run:
        epochs = total_epochs_override or default_epochs

        print(f"\n{'━' * 70}")
        print(f"  Model: {tag}  |  {n_folds}-Fold CV")
        print(f"{'━' * 70}")

        oof_probs  = np.zeros((len(df_train), NUM_CLASSES), dtype=np.float32)
        oof_filled = np.zeros(len(df_train), dtype=bool)

        fold_test_probs = []
        fold_val_f1s    = []

        for fold, (train_img_idx, val_img_idx) in enumerate(
            skf.split(img_info, img_info["stratum"])
        ):
            print(f"\n  ── Fold {fold + 1}/{n_folds} ──")
            train_imgs = img_info.iloc[train_img_idx]["image_id"].values
            val_imgs   = img_info.iloc[val_img_idx]["image_id"].values

            train_df = df_train[df_train["image_id"].isin(train_imgs)].reset_index(drop=True)
            val_df   = df_train[df_train["image_id"].isin(val_imgs)].reset_index(drop=True)

            print(f"  Train: {len(train_df):,} instances ({len(train_imgs)} images) | "
                  f"Val: {len(val_df):,} instances ({len(val_imgs)} images)")

            for ct in ["orb", "tur"]:
                v = (val_df["image_id"].apply(extract_cam_type) == ct).sum()
                t = (train_df["image_id"].apply(extract_cam_type) == ct).sum()
                print(f"    {ct}: train={t}, val={v}")

            for cls_id, name in enumerate(CLASS_NAMES):
                n_cls = (val_df["class_id"] == cls_id).sum()
                flag = " ⚠" if n_cls < 20 else ""
                print(f"    {cls_id}:{name:<22} {n_cls:4d}{flag}")

            class_weights = compute_class_weights(train_df["class_id"].values)

            fold_tag_suffix = f"_fold{fold + 1}"

            model, val_probs, val_labels, best_f1 = train_model(
                model_key=model_key,
                tag=tag,
                img_size=img_size,
                batch_size=batch_size,
                total_epochs=epochs,
                train_df=train_df,
                val_df=val_df,
                img_dir=img_dir,
                set_label=set_label,
                class_weights=class_weights,
                resume=resume,
                resume_if_exists=resume_if_exists,
                tag_suffix=fold_tag_suffix,
            )

            val_original_idx = df_train[df_train["image_id"].isin(val_imgs)].index
            oof_probs[val_original_idx]  = val_probs
            oof_filled[val_original_idx] = True

            fold_val_f1s.append(best_f1)

            print(f"\n  Fold {fold + 1} test inference (TTA)...")
            if TTA_ENABLED:
                test_probs, row_ids = get_tta_probs(model, df_test, TEST_DIR, img_size, batch_size)
            else:
                test_ds = PigPostureDataset(
                    df_test, TEST_DIR, get_val_transform(img_size), img_size, mode="test"
                )
                test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                                     num_workers=NUM_WORKERS, pin_memory=True)
                test_probs, row_ids = get_probs(model, test_dl, with_ids=True)

            fold_test_probs.append((test_probs, best_f1))

            del model
            gc.collect()
            torch.cuda.empty_cache()

        if oof_filled.all():
            oof_preds = oof_probs.argmax(axis=1)
            oof_f1    = f1_score(
                df_train["class_id"].values, oof_preds,
                average="macro", zero_division=0,
            )
            oof_per_c = f1_score(
                df_train["class_id"].values, oof_preds,
                average=None, zero_division=0,
            )
            print(f"\n{'─' * 60}")
            print(f"  {set_label} {tag} → OOF Macro F1: {oof_f1:.4f}")
            print(
                f"    Sit={oof_per_c[2]:.3f} Left={oof_per_c[0]:.3f} "
                f"Right={oof_per_c[1]:.3f} Stand={oof_per_c[3]:.3f} "
                f"Stern={oof_per_c[4]:.3f}"
            )
            print(f"  Fold F1s: {[f'{f:.4f}' for f in fold_val_f1s]}")
            print(f"  Mean fold F1: {np.mean(fold_val_f1s):.4f} ± {np.std(fold_val_f1s):.4f}")
            print(f"{'─' * 60}")
        else:
            missing = (~oof_filled).sum()
            print(f"  ⚠ OOF incomplete: {missing} rows missing")
            oof_f1 = np.mean(fold_val_f1s)

        sub_dir = OUT_DIR / f"submissions_{set_label}"
        sub_dir.mkdir(exist_ok=True)

        for i, (fp, fw) in enumerate(fold_test_probs):
            save_submission(
                fp, row_ids,
                sub_dir / f"{set_label}_{tag}_fold{i + 1}_tta.csv",
                set_label, f"{tag} Fold {i+1} (TTA)",
            )

        ens_probs = weighted_ensemble(fold_test_probs)
        ens_path  = sub_dir / f"{set_label}_{tag}_kfold_ensemble.csv"
        save_submission(ens_probs, row_ids, ens_path, set_label,
                        f"{tag} {n_folds}-Fold Weighted Ensemble (OOF F1={oof_f1:.4f})")

        save_submission(ens_probs, row_ids, "submission.csv")

        print(f"\n✓ {set_label} {tag} K-Fold complete! OOF F1: {oof_f1:.4f}")

    print(f"\n✓ {set_label} pipeline complete!")
    return ens_path


def inference_only(train_csv, img_dir, set_label, n_folds=N_FOLDS):
    print(f"\n{'#' * 70}")
    print(f"#  {set_label} → INFERENCE ONLY ({n_folds}-fold)")
    print(f"{'#' * 70}")

    df_test  = pd.read_csv(TEST_CSV)
    sub_dir  = OUT_DIR / f"submissions_{set_label}"
    sub_dir.mkdir(exist_ok=True)

    for model_key, tag, img_size, batch_size, _ in MODEL_CONFIGS:
        fold_test_probs = []
        row_ids_global  = None

        for fold in range(1, n_folds + 1):
            fold_tag      = f"{tag}_fold{fold}"
            best_ema_path = get_best_ema_path(set_label, fold_tag)

            if not best_ema_path.exists():
                print(f"  ⚠ [{fold_tag}] checkpoint not found, skipping")
                continue

            print(f"\n  [{fold_tag}] loading...")
            if is_dinov2(model_key):
                model = timm.create_model(
                    DINOV2_TIMM_MAP[model_key], pretrained=False,
                    num_classes=NUM_CLASSES, img_size=224,
                )
            else:
                model = timm.create_model(
                    model_key, pretrained=False, num_classes=NUM_CLASSES,
                    drop_rate=0.3, drop_path_rate=0.2,
                )
            model.load_state_dict(
                torch.load(best_ema_path, map_location=DEVICE, weights_only=False)
            )
            model = model.to(DEVICE).eval()

            if TTA_ENABLED:
                test_probs, row_ids = get_tta_probs(model, df_test, TEST_DIR, img_size, batch_size)
            else:
                test_ds = PigPostureDataset(
                    df_test, TEST_DIR, get_val_transform(img_size), img_size, mode="test"
                )
                test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                                     num_workers=NUM_WORKERS, pin_memory=True)
                test_probs, row_ids = get_probs(model, test_dl, with_ids=True)

            if row_ids_global is None:
                row_ids_global = row_ids
            fold_test_probs.append((test_probs, 1.0))

            save_submission(test_probs, row_ids_global,
                            sub_dir / f"{set_label}_{fold_tag}_tta.csv",
                            set_label, f"{fold_tag} (TTA)")
            del model
            gc.collect()
            torch.cuda.empty_cache()

        if not fold_test_probs:
            print(f"✗ [{tag}] No checkpoints found!")
            continue

        ens      = weighted_ensemble(fold_test_probs)
        ens_path = sub_dir / f"{set_label}_{tag}_kfold_ensemble.csv"
        save_submission(ens, row_ids_global, ens_path, set_label,
                        f"{tag} {len(fold_test_probs)}-Fold Ensemble")
        save_submission(ens, row_ids_global, "submission.csv")

    return ens_path


class Args:
    resume           = False
    resume_if_exists = False
    inference_only   = False
    set              = "T2"
    tags             = None
    total_epochs     = 22
    n_folds          = 3


def main():
    args = Args()
    print("=" * 70)
    print("Pig Posture SOTA v5")
    print("  + LLRD (layer-wise LR decay)")
    print("  + Confusion-calibrated penalty loss")
    print("  + Class-specific augmentation")
    print("  + Camera-type-aware K-Fold stratification")
    print("  + K-Fold CV (OOF Ensemble) -- pseudo-labeling removed")
    print("=" * 70)
    print(f"\nModels  : {[c[1] for c in MODEL_CONFIGS]}")
    print(f"Set     : {args.set}")
    print(f"K-Folds : {args.n_folds}")
    print(f"Resume  : {args.resume or args.resume_if_exists}")
    print(f"TTA     : {TTA_ENABLED} (8x)")
    print(f"LLRD    : decay={LLRD_DECAY}")

    sets_to_run = {
        "T1":   [("T1", TRAIN1_CSV, TRAIN1_DIR)],
        "T2":   [("T2", TRAIN2_CSV, TRAIN2_DIR)],
        "both": [("T1", TRAIN1_CSV, TRAIN1_DIR), ("T2", TRAIN2_CSV, TRAIN2_DIR)],
    }[args.set]

    set_seed(SEED)
    submission_paths = []

    for set_label, train_csv, img_dir in sets_to_run:
        if args.inference_only:
            path = inference_only(train_csv, img_dir, set_label, n_folds=args.n_folds)
        else:
            path = run_kfold_pipeline(
                train_csv=train_csv,
                img_dir=img_dir,
                set_label=set_label,
                total_epochs_override=args.total_epochs,
                resume=args.resume,
                resume_if_exists=args.resume_if_exists,
                specific_tags=args.tags,
                n_folds=args.n_folds,
            )
        if path:
            submission_paths.append(path)
        gc.collect()
        torch.cuda.empty_cache()

    print("\n" + "=" * 70)
    print("DONE")
    print("=" * 70)
    if submission_paths:
        print("\n✓ Submission files:")
        for p in submission_paths:
            print(f"   {p}")
    print("\nNote: The winner is determined by the T2 submission.")
    print("     OOF F1 score is the most reliable estimate of true performance.")


if __name__ == "__main__":
    main()

Pig Posture SOTA v5
  + LLRD (layer-wise LR decay)
  + Confusion-calibrated penalty loss
  + Class-specific augmentation
  + Camera-type-aware K-Fold stratification
  + K-Fold CV (OOF Ensemble) -- pseudo-labeling removed

Models  : ['dinov2_g']
Set     : T2
K-Folds : 3
Resume  : False
TTA     : True (8x)
LLRD    : decay=0.85

######################################################################
#  T2 K-FOLD PIPELINE  (v5, 3-fold)
######################################################################

Training instances: 23,450
  0: Lateral_lying_left         3083 (13.1%)
  1: Lateral_lying_right        3435 (14.6%)
  2: Sitting                     695 (3.0%)
  3: Standing                   9928 (42.3%)
  4: Sternal_lying              6309 (26.9%)

Total images: 3,150 | Strata count: 9

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Model: dinov2_g  |  3-Fold CV
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ── Fold 1/3 ──
  Train: 1

model.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

  [DINOv2] vit_giant_patch14_reg4_dinov2.lvd142m → 1134.8M param
  LLRD head: lr=5.0e-05
  LLRD blocks: 40 blocks | last=4.25e-05 → first=7.51e-08
  LLRD patch/norm: lr=5.43e-08
  LLRD total trainable: 1134.8M params, 42 groups
  ✓ Starting from scratch (epoch 1/22)


  train:   0%|                                                                       | 0/1950 [00:00<?, ?it/s]

  val  :   0%|                                                                        | 0/982 [00:00<?, ?it/s]

  Ep 01/22 | lr=1.7e-05 | tr_loss=0.2451 acc=0.715 | val_F1=0.3725 | GPU 22.9/42GB | 538s
    Sit=0.292 Left=0.342 Right=0.037 Stand=0.678 Stern=0.515
    ★ Best val F1=0.3725
    ✓ Checkpoint saved: T2_dinov2_g_fold1_full_ckpt.pt


  train:   0%|                                                                       | 0/1950 [00:00<?, ?it/s]

  val  :   0%|                                                                        | 0/982 [00:00<?, ?it/s]

  Ep 02/22 | lr=3.3e-05 | tr_loss=0.1094 acc=0.889 | val_F1=0.6488 | GPU 27.4/42GB | 536s
    Sit=0.583 Left=0.487 Right=0.552 Stand=0.925 Stern=0.698
    ★ Best val F1=0.6488
    ✓ Checkpoint saved: T2_dinov2_g_fold1_full_ckpt.pt


  train:   0%|                                                                       | 0/1950 [00:00<?, ?it/s]

In [35]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import random

set_seed(SEED)

VIZ_SET  = "T2"

if VIZ_SET == "T2":
    train_csv_viz, img_dir_viz = TRAIN2_CSV, TRAIN2_DIR
else:
    train_csv_viz, img_dir_viz = TRAIN1_CSV, TRAIN1_DIR

df_all   = pd.read_csv(train_csv_viz)
img_info = build_img_info(df_all)

skf_viz  = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits   = list(skf_viz.split(img_info, img_info["stratum"]))
_, val_idx = splits[0]
val_imgs = img_info.iloc[val_idx]["image_id"].values
val_df_viz = df_all[df_all["image_id"].isin(val_imgs)].reset_index(drop=True)

model_key, tag, img_size, batch_size, _ = MODEL_CONFIGS[0]
ckpt_path = get_best_ema_path(VIZ_SET, f"{tag}_fold1")

viz_model = timm.create_model(
    DINOV2_TIMM_MAP[model_key], pretrained=False,
    num_classes=NUM_CLASSES, img_size=224,
)
viz_model.load_state_dict(
    torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
)
viz_model = viz_model.to(DEVICE).eval()

val_tf = get_val_transform(img_size)

img_counts      = val_df_viz.groupby("image_id").size()
candidate_imgs  = img_counts[img_counts >= 2].index.tolist()

if len(candidate_imgs) < 4:
    candidate_imgs = val_df_viz["image_id"].unique().tolist()
chosen = random.sample(candidate_imgs, 4)

def _run_inference(image_id):
    img_path = img_dir_viz / image_id
    full_img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    if full_img is None:
        return None, []
    full_img = cv2.cvtColor(full_img, cv2.COLOR_BGR2RGB)
    preds = []
    for _, row in val_df_viz[val_df_viz["image_id"] == image_id].iterrows():
        x, y, w, h = parse_bbox(row["bbox"])
        crop   = crop_and_resize(img_path, x, y, w, h, img_size, pad=PAD_FRACTION)
        tensor = val_tf(image=crop)["image"].unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs = torch.softmax(viz_model(tensor), dim=1)[0].cpu().numpy()
        idx  = int(probs.argmax())
        conf = float(probs[idx])
        preds.append({
            "bbox":  (int(x), int(y), int(w), int(h)),
            "label": CLASS_NAMES[idx].lower().replace("_", " "),
            "conf":  conf,
        })
    return full_img, preds

def _annotate(img, preds):
    out   = img.copy()
    color = (20, 180, 70)
    for p in preds:
        x, y, w, h = p["bbox"]
        text = f"{p['label']}  {p['conf']:.2f}"
        cv2.rectangle(out, (x, y), (x + w, y + h), color, 3)
        (tw, th), bl = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.85, 2)
        cv2.rectangle(out, (x, y - th - bl - 8), (x + tw + 8, y), color, cv2.FILLED)
        cv2.putText(out, text, (x + 4, y - bl - 3),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255, 255, 255), 2, cv2.LINE_AA)
    return out

all_results = [_run_inference(iid) for iid in chosen]

DPI     = 100
W_PX    = 1920
PADDING = 18
TITLE_H = 50
L, R    = 0.01, 0.99
WSPACE  = 0.02
BG      = "white"

img_area_w_px = int(W_PX * (R - L - WSPACE) / 2)

for fig_idx in range(2):
    pair      = [chosen[fig_idx * 2], chosen[fig_idx * 2 + 1]]
    pair_res  = [all_results[fig_idx * 2], all_results[fig_idx * 2 + 1]]

    max_display_h = 0
    for full_img, _ in pair_res:
        if full_img is not None:
            h, w = full_img.shape[:2]
            max_display_h = max(max_display_h, int(h * img_area_w_px / w))

    H_PX = max_display_h + TITLE_H + 2 * PADDING

    content_frac = max_display_h / H_PX
    title_frac   = TITLE_H / H_PX
    pad_frac     = PADDING / H_PX

    fig, axes = plt.subplots(
        1, 2,
        figsize=(W_PX / DPI, H_PX / DPI),
        facecolor=BG,
        gridspec_kw={"wspace": WSPACE},
    )
    fig.subplots_adjust(
        left=L, right=R,
        bottom=pad_frac,
        top=1 - pad_frac - title_frac,
    )

    for ax, image_id, (full_img, preds) in zip(axes, pair, pair_res):
        ax.set_facecolor(BG)
        if full_img is not None:
            ax.imshow(_annotate(full_img, preds), aspect="equal")
        else:
            ax.text(0.5, 0.5, "image not found",
                    ha="center", va="center", color="#333333",
                    transform=ax.transAxes, fontsize=18)
        ax.set_title(
            image_id.lower(),
            color="#111111", fontsize=16,
            fontfamily="monospace", pad=10, fontweight="bold",
        )
        ax.axis("off")

    out_path = f"validation_predictions_{fig_idx + 1}.png"
    plt.savefig(out_path, dpi=DPI, facecolor=BG)
    plt.close(fig)
    print(f"saved → {out_path}  ({W_PX}×{H_PX} px)")

saved → validation_predictions_1.png  (1920×604 px)
saved → validation_predictions_2.png  (1920×604 px)
